# Ethylbenzene: step-by-step assignment test in JASON

A hand-driven experiment to find out what JASON displays when we assign
shifts to specific signals, one step at a time.

**How it works**

1. **Inspect** lists the molecule's atoms, JASON's predicted shifts, and every peak/multiplet with its JEOL ID.
2. You copy the JEOL IDs you want into the **assignment table** (one row = one shift → one signal).
3. **run_step(n)** copies the original `.jjh5` to `ethylbenzene_step<n>.jjh5`, applies every row with
   `step <= n`, and saves it. Open that file in JASON and note what is displayed.
   `run_step(n, tag="b")` saves a variant as `ethylbenzene_step<n>b.jjh5` without overwriting.

Each step starts from a fresh copy of the original, so steps never affect each other.

| step | adds | result |
|---|---|---|
| 1 | C8 (CH₃) and C2 (quat) carbons → 1D 13C peaks; H8 → HSQC peak (1H axis) | ✓ values shown; HMBC CH₃ → C2 line appears by itself |
| 2 | explicit HMBC H8 → C2 assignment | not needed (line already drawn); not run |
| 3 | C7 (CH₂) carbon → 1D 13C peak; H7 → HSQC peak | ✓ HMBC CH₃ → C7 and COSY CH₂–CH₃ appear; HMBC CH₂ → C2 missing |
| 3b | H7 → HSQC **multiplet** instead of peak | ✗ rejected: "Only NMRPeak objects can be assigned from a 2D spectrum" |
| 4 | aromatic carbons → 1D 13C peaks; their protons → HSQC peaks | to test |

Atom numbers (from the molfile): 1, 3 ortho · 2 quaternary · 4, 6 meta · 5 para · 7 CH₂ · 8 CH₃.

### What we have learnt so far

* Carbons are assigned to **1D 13C peaks**; protons to **HSQC peaks** on the 1H axis.
* Pure-shift peaks are rejected ("Cannot find a matching merged 1D signal"): in this document they are in no merged signal.
* 2D multiplets are rejected; only 2D *peaks* can be assigned.
* HMBC/COSY lines are drawn by JASON itself once both ends (merged signals) carry an assigned shift.
* HMBC CH₂ → C2 stays missing: its CH₂ end is a merged signal holding only the HSQC multiplet, which can't be assigned.


In [34]:
from pathlib import Path

# The document as JASON exported it (use the backup from a run, never the live input.jjh5)
SOURCE = Path("/Users/vsmw51/Documents/JASON/External Tools/simpleNMRverify/"
              "2026-10-05T18-57-21.380_60447/input.20261005-185758.bak.jjh5")
OUT_DIR = Path.home() / "Documents" / "ethylbenzene_tests"
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [35]:
import shutil
import beautifuljason as bjason

H1 = bjason.Molecule.Atom.NuclType.H1
C13 = bjason.Molecule.Atom.NuclType.C13

print("beautifuljason", bjason.__version__)
print("set_assignments available:", hasattr(bjason.Molecule, "set_assignments"))


def signals_of(spectrum):
    """Every peak and multiplet in a spectrum, as (kind, object)."""
    out = []
    for kind in ("peaks", "multiplets"):
        try:
            out += [(kind[:-1], obj) for obj in getattr(spectrum, kind)]
        except KeyError:            # this spectrum has no table of that kind
            pass
    return out


def fmt(values):
    return "(" + ", ".join(f"{float(v):.4f}" for v in values) + ")"

beautifuljason 1.3.0b1
set_assignments available: True


## 1 · Inspect the document
Atom numbers are 1-based (as in simpleNMR and the molfile); beautifuljason counts atoms from 0.

In [36]:
def experiment_name(spectrum):
    """HSQC, HMBC, COSY, C13_1D, H1_1D, PURESHIFT, ... using the same rules as json_converter."""
    info = spectrum.spec_info
    expt = bjason.utils.ensure_str(info.get_param("ExperimentType.str") or "")
    pulprog = bjason.utils.ensure_str(info.get_param("PulseProgram") or "").lower()
    nuclides = tuple(info.nuclides)

    print("expt:", expt)
    if nuclides == ("1H",) and any(k in pulprog for k in ("psyche", "pure", "zangger")):
        return "PURESHIFT"
    if expt == "Conventional pulse acquire":
        return {("1H",): "H1_1D", ("13C",): "C13_1D"}.get(nuclides, expt)
    if expt == "Unknown" and pulprog == "udeft":
        return {("13C",): "C13_1D"}.get(nuclides, expt)
    if expt == "generic COSY":
        return "COSY"
    return expt or "unknown"

with bjason.Document(str(SOURCE), mode="r") as doc:
    mol = doc.mol_data[0]

    print("ATOMS  (atomNumber, element, attached H)")
    for i, atom in enumerate(mol.atoms):
        print(f"  {i + 1:3d}  {atom.type.name:2}  nH={atom.nh}")

    print("\nPREDICTED SHIFTS  (nucleus, atomNumber(s), mark, predicted ppm)")
    for spectrum in mol.spectra:
        for shift in spectrum.shifts:
            value, _ = shift.get_value_error_pair(bjason.Molecule.CalcMethod.Best)
            atoms = ",".join(str(int(n) + 1) for n in shift.nums)
            print(f"  {spectrum.nucleus.name:4} atom {atoms:6} mark={shift.mark!s:5} {float(value):9.4f}")

    print("\nSIGNALS  (spectrum index, experiment, nuclides, kind, JEOL ID, position)")
    for s_index, spectrum in enumerate(doc.nmr_data):
        nuclides = tuple(spectrum.spec_info.nuclides)
        expt = experiment_name(spectrum)
        for kind, obj in signals_of(spectrum):
            print(f"  [{s_index}] {expt:10} {str(nuclides):16} {kind:9} {obj.id}  {fmt(obj.pos)}")


    for s_index, spectrum in enumerate(doc.nmr_data):
        info = spectrum.spec_info
        print(s_index, info.nuclides,
              bjason.utils.ensure_str(info.get_param("ExperimentType.str")),
              bjason.utils.ensure_str(info.get_param("PulseProgram")))
    # si = spectrum.spec_info
    # print(list(si.h5_group.attrs.keys()))   # all parameter names
    # print(list(si.h5_group.keys()))         # sub-groups (e.g. 'Nucleides')

    # for name in si.h5_group.attrs.keys():
    #     print(f"{name:28} {bjason.utils.ensure_str(si.get_param(name))!r}")

ATOMS  (atomNumber, element, attached H)
    1  C   nH=1
    2  C   nH=0
    3  C   nH=1
    4  C   nH=1
    5  C   nH=1
    6  C   nH=1
    7  C   nH=2
    8  C   nH=3

PREDICTED SHIFTS  (nucleus, atomNumber(s), mark, predicted ppm)
  C13  atom 1      mark=None   127.7333
  C13  atom 2      mark=None   143.9667
  C13  atom 3      mark=None   127.7333
  C13  atom 4      mark=None   128.2667
  C13  atom 5      mark=None   125.6000
  C13  atom 6      mark=None   128.2667
  C13  atom 7      mark=None    29.4000
  C13  atom 8      mark=None    15.7000
  H1   atom 1      mark=None     7.1000
  H1   atom 3      mark=None     7.1000
  H1   atom 4      mark=None     7.1000
  H1   atom 5      mark=None     7.1000
  H1   atom 6      mark=None     7.1000
  H1   atom 7      mark=None     2.4500
  H1   atom 8      mark=None     1.0800

SIGNALS  (spectrum index, experiment, nuclides, kind, JEOL ID, position)
expt: Conventional pulse acquire
expt: Unknown
  [1] C13_1D     ('13C',)         peak      {

In [37]:
with bjason.Document(str(SOURCE), mode="r") as doc:
    for s_index, spectrum in enumerate(doc.nmr_data):
        if experiment_name(spectrum) != "HSQC":
            continue
        for kind, obj in signals_of(spectrum):
            if kind == "multiplet":
                print(f"  [{s_index}] {obj.id}  range={[list(map(float, r)) for r in obj.range]}")

expt: Conventional pulse acquire
expt: Unknown
expt: generic COSY
expt: Unknown
expt: HMBC
expt: Unknown
expt: HSQC
  [6] {acbe2c69-58b9-4bf0-bfeb-050d538ea171}  range=[[7.388253079829015, 7.4118193338862595], [128.26067736601007, 128.55034054175454], [0.0, 0.0]]
  [6] {bb4861a6-78c2-48fa-930f-913298c250d2}  range=[[7.311085149876865, 7.331416820043898], [127.77580639791609, 128.10325172701852], [0.0, 0.0]]
  [6] {9afcb99b-a405-4c22-8cd2-a009c8b720a1}  range=[[7.281973894864977, 7.307850565986656], [125.55295637535532, 125.79224334662248], [0.0, 0.0]]
  [6] {23c65e02-8b2e-4fba-b64e-07d13d4e7a0a}  range=[[2.7610954847580738, 2.786256376219939], [28.92090292637715, 29.19734728108532], [0.0, 0.0]]
  [6] {a70e7b87-3667-4ff8-9ff6-8f3f42ab1e5f}  range=[[1.348115687180645, 1.3829084824052538], [15.576318530159885, 15.904860208081283], [0.0, 0.0]]


In [38]:
with bjason.Document(str(SOURCE), mode="r") as doc:
    mol = doc.mol_data[0]
    print(mol.spectra)
    for spectrum in mol.spectra:
        print(spectrum.spec_type)

    print(dir(spectrum))

1
1
['List', 'Shift', 'Type', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__firstlineno__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__static_attributes__', '__str__', '__subclasshook__', '__weakref__', '_attr', 'get_or_create_shift', 'h5_group', 'nucleus', 'origin', 'shifts', 'spec_type']


## 2 · The assignment table

One row = one predicted shift → one signal. Copy each `jeol_id` from the SIGNALS list above.

* `atom` – atomNumber (1-based); `nucleus` – `"C"` or `"H"`; `mark` – the CH₂ proton mark from the
  PREDICTED SHIFTS list (`None` when there is only one shift on that atom)
* `dim` – `None` for a 1D signal; for a 2D peak, the axis this shift sits on (0 = first nuclide listed)
* `expect` – the position I expect on that axis (from the earlier diagnostic); only used as a check
* `calc` – calculated value, for reference only

In [ ]:
ASSIGNMENTS = [
    # ---- step 1: two carbons from the 1D 13C, the CH3 proton from the HSQC peak (1H axis)
    dict(step=1, label="C8 carbon (CH3)",   atom=8, nucleus="C", mark=None, jeol_id="{84fce28e-910a-4c13-bf36-b2839f6ad45a}", dim=None, expect=15.7155,  calc=15.7),
    dict(step=1, label="C2 carbon (quat)",  atom=2, nucleus="C", mark=None, jeol_id="{db5d5e8d-a444-4c9f-8652-052226a52d1a}", dim=None, expect=144.3054, calc=144.0),
    dict(step=1, label="H8 proton (CH3)",   atom=8, nucleus="H", mark=None, jeol_id="{49dad087-f74e-45da-ada3-1bd45abb1bbc}", dim=0,    expect=1.37,     calc=None),

    # ---- step 2 (not needed - JASON draws the HMBC line itself): explicit HMBC H8 -> C2
    # dict(step=2, label="HMBC H8 -> C2 (H)", atom=8, nucleus="H", mark=None, jeol_id="PASTE-ID", dim=0, expect=1.3680,   calc=None),
    # dict(step=2, label="HMBC H8 -> C2 (C)", atom=2, nucleus="C", mark=None, jeol_id="PASTE-ID", dim=1, expect=144.4002, calc=None),

    # ---- step 3: the CH2 - carbon from the 1D 13C, proton from the HSQC peak (1H axis)
    dict(step=3, label="C7 carbon (CH2)",   atom=7, nucleus="C", mark=None, jeol_id="{60d8daa1-0725-4220-9bc5-0ed42dc2e803}", dim=None, expect=28.9897,  calc=29.4),
    dict(step=3, label="H7 proton (CH2)",   atom=7, nucleus="H", mark=None, jeol_id="{bfb0db5f-1680-4585-aa52-955107815816}", dim=0,    expect=2.7775,   calc=None),
    # step 3b, rejected ("Only NMRPeak objects can be assigned from a 2D spectrum"): H7 -> HSQC multiplet
    # dict(step=3, label="H7 proton (CH2)", atom=7, nucleus="H", mark=None, jeol_id="{a70e7b87-3667-4ff8-9ff6-8f3f42ab1e5f}", dim=0, expect=2.78, calc=None),

    # ---- step 4: the aromatic ring - carbons from the 1D 13C, protons from the HSQC peaks (1H axis)
    # Symmetric pairs (ortho 1/3, meta 4/6) share one 1D peak and one HSQC peak, so both rows of a pair
    # get the same jeol_id. If PREDICTED SHIFTS shows one shift for both atoms (e.g. "atom 1,3"),
    # run_step uses it once and reports the second row as "same shift".
    dict(step=4, label="C1 carbon (ortho)", atom=1, nucleus="C", mark=None, jeol_id="{981e8a05-cc8d-479a-a660-57d1de40ed90}", dim=None, expect=127.9292, calc=127.7),
    dict(step=4, label="C3 carbon (ortho)", atom=3, nucleus="C", mark=None, jeol_id="{981e8a05-cc8d-479a-a660-57d1de40ed90}", dim=None, expect=127.9292, calc=127.7),
    dict(step=4, label="H1 proton (ortho)", atom=1, nucleus="H", mark=None, jeol_id="{b02fad61-a90c-43ae-9115-ebbc664d9306}", dim=0,    expect=7.3226,   calc=None),
    dict(step=4, label="H3 proton (ortho)", atom=3, nucleus="H", mark=None, jeol_id="{b02fad61-a90c-43ae-9115-ebbc664d9306}", dim=0,    expect=7.3226,   calc=None),

    dict(step=4, label="C4 carbon (meta)",  atom=4, nucleus="C", mark=None, jeol_id="{e0ce5f38-c916-452d-ac68-55a58caf9cc6}", dim=None, expect=128.3836, calc=128.3),
    dict(step=4, label="C6 carbon (meta)",  atom=6, nucleus="C", mark=None, jeol_id="{e0ce5f38-c916-452d-ac68-55a58caf9cc6}", dim=None, expect=128.3836, calc=128.3),
    dict(step=4, label="H4 proton (meta)",  atom=4, nucleus="H", mark=None, jeol_id="{fc3bd16a-e67c-47d6-bc1d-73daf2d26bc1}", dim=0,    expect=7.4012,   calc=None),
    dict(step=4, label="H6 proton (meta)",  atom=6, nucleus="H", mark=None, jeol_id="{fc3bd16a-e67c-47d6-bc1d-73daf2d26bc1}", dim=0,    expect=7.4012,   calc=None),

    dict(step=4, label="C5 carbon (para)",  atom=5, nucleus="C", mark=None, jeol_id="{7dd23078-a81f-4aca-a9e8-ceef2c501e24}", dim=None, expect=125.6658, calc=125.6),
    dict(step=4, label="H5 proton (para)",  atom=5, nucleus="H", mark=None, jeol_id="{f80dacb4-d950-47eb-8ad9-7a8b46596427}", dim=0,    expect=7.2956,   calc=None),

#  step 6



]


## 3 · Helpers

In [47]:
MOVES = [
    # step, label, JEOL ID, {axis: new position}
    dict(step=5, label="HMBC meta-H -> C2", jeol_id="{1774e688-a307-4f0c-92ad-b60b6da1c87b}", new={0: 7.4012, 1: 144.3054}),
        # ---- step 6: align every remaining HMBC/COSY peak with the assigned 1D 13C / HSQC positions
    dict(step=6, label="HMBC CH3 -> C2",     jeol_id="{bb84d936-9adf-44bd-8c46-ebd0a7aad069}", new={0: 1.3676, 1: 144.3054}),  # from (1.3680, 144.4002)
    dict(step=6, label="HMBC CH3 -> C7",     jeol_id="{17aa3cdf-e8f8-478b-b07c-124e637c561c}", new={0: 1.3676, 1: 28.9897}),   # from (1.3681, 28.9667)
    dict(step=6, label="HMBC CH2 -> C2",     jeol_id="{9cb4a1c8-fb58-4864-a418-a446a0803e6e}", new={0: 2.7775, 1: 144.3054}),  # from (2.7748, 144.3940)
    dict(step=6, label="HMBC CH2 -> ortho",  jeol_id="{82292e56-7132-4af1-a2df-269c3383b84b}", new={0: 2.7775, 1: 127.9292}),  # from (2.7752, 127.9209)
    dict(step=6, label="HMBC para-H -> ortho", jeol_id="{00a416e8-712d-48b2-b645-73536f816579}", new={0: 7.2956, 1: 127.9292}),  # from (7.2955, 127.9546)
    dict(step=6, label="COSY CH2 - CH3",     jeol_id="{c722cbc6-e190-4a44-b790-757bcb2691ab}", new={0: 2.7775, 1: 1.3676}),    # from (2.7759, 1.3684)
    dict(step=6, label="COSY meta - ortho",  jeol_id="{0ce9e917-3c5f-4d11-8297-93b930ab33fd}", new={0: 7.4012, 1: 7.3226}),    # from (7.4051, 7.3263)
    dict(step=6, label="COSY meta - para",   jeol_id="{e48c637d-1420-4c76-941c-39c6c30dd5a8}", new={0: 7.4012, 1: 7.2956}),    # from (7.4047, 7.2966)
]

def apply_moves(doc, step):
    for m in [m for m in MOVES if m["step"] <= step]:
        peak, s_index = find_signal(doc, m["jeol_id"])
        pos = peak.h5_group.attrs["Pos"].copy()
        print(f"  move {m['label']}: {fmt(pos)}", end="")
        for axis, value in m["new"].items():
            pos[axis] = value
        peak.h5_group.attrs["Pos"] = pos
        print(f" -> {fmt(pos)}   ShapePar={peak.h5_group.attrs.get('ShapePar')}")


def find_shift(mol, nucleus, atom, mark):
    """The predicted shift for this nucleus on atom (1-based), with this CH2 mark."""
    nucl = C13 if nucleus == "C" else H1
    for spectrum in mol.spectra:
        if spectrum.nucleus != nucl:
            continue
        for shift in spectrum.shifts:
            if atom - 1 in [int(n) for n in shift.nums] and (mark is None or shift.mark == mark):
                return shift
    raise LookupError(f"no predicted {nucleus} shift on atom {atom} with mark {mark}")


def find_signal(doc, jeol_id):
    """The peak or multiplet with this JEOL ID, and the index of its spectrum."""
    for s_index, spectrum in enumerate(doc.nmr_data):
        for kind, obj in signals_of(spectrum):
            if str(obj.id) == jeol_id:
                return obj, s_index
    raise LookupError(f"no peak or multiplet with ID {jeol_id}")

def run_step(step, tag=""):
    """Copy SOURCE, apply all rows with row['step'] <= step, save as ethylbenzene_step<step>.jjh5."""
    rows = [r for r in ASSIGNMENTS if r["step"] <= step]
    missing = [r["label"] for r in rows if r["jeol_id"] == "PASTE-ID"]
    if missing:
        print("Fill in jeol_id for:", ", ".join(missing))
        return

    out = OUT_DIR / f"ethylbenzene_step{step}{tag}.jjh5"
    shutil.copy2(SOURCE, out)

    with bjason.Document(str(out), mode="r+") as doc:
        mol = doc.mol_data[0]
        apply_moves(doc, step)
        assignments = []
        print(f"Step {step}: {len(rows)} assignment(s)")
        for r in rows:
            shift = find_shift(mol, r["nucleus"], r["atom"], r["mark"])
            signal, s_index = find_signal(doc, r["jeol_id"])
            position = float(signal.pos[0 if r["dim"] is None else r["dim"]])
            check = "ok" if abs(position - r["expect"]) < 0.05 else f"CHECK: expected {r['expect']}"
            new = (shift, signal) if r["dim"] is None else (shift, signal, r["dim"])
            # a symmetric pair may share one predicted shift: use it once
            if any(a[0].h5_group.name == shift.h5_group.name and a[1].id == signal.id for a in assignments):
                print(f"  {r['label']:20} -> same shift and signal as an earlier row, used once")
                continue
            print(f"  {r['label']:20} -> spectrum [{s_index}] at {position:9.4f}  {check}")
            assignments.append(new)

        try:
            mol.set_assignments(assignments)
        except Exception as e:                  # JASON's rule checks raise here
            print(f"\nJASON rejected step {step}: {e}")
            failed = True
        else:
            failed = False

    if failed:
        out.unlink()
        print("No file written.")
    else:
        print(f"\nSaved {out} - open it in JASON.")

## 4 · Run the steps
Run one cell per step, open the file in JASON, and note what is displayed before moving on.

In [41]:
run_step(1)

Step 1: 3 assignment(s)
  C8 carbon (CH3)      -> spectrum [1] at   15.7155  ok
  C2 carbon (quat)     -> spectrum [1] at  144.3054  ok
  H8 proton (CH3)      -> spectrum [6] at    1.3676  ok

Saved /Users/vsmw51/Documents/ethylbenzene_tests/ethylbenzene_step1.jjh5 - open it in JASON.


In [42]:
# run_step(2)    # not needed

In [43]:
run_step(3)

Step 3: 5 assignment(s)
  C8 carbon (CH3)      -> spectrum [1] at   15.7155  ok
  C2 carbon (quat)     -> spectrum [1] at  144.3054  ok
  H8 proton (CH3)      -> spectrum [6] at    1.3676  ok
  C7 carbon (CH2)      -> spectrum [1] at   28.9897  ok
  H7 proton (CH2)      -> spectrum [6] at    2.7775  ok

Saved /Users/vsmw51/Documents/ethylbenzene_tests/ethylbenzene_step3.jjh5 - open it in JASON.


In [44]:
# run_step(3, tag="b")    # H7 -> HSQC multiplet: rejected

### Step 4 · the aromatic ring

What the earlier merged-signal diagnostic predicts:

| group | 1D 13C peak in | HSQC peak in |
|---|---|---|
| ortho | M6 | M6 (same group) |
| meta | M8 | M9 (different groups) |
| para | M4 | M5 (different groups) |

* All five carbons and their protons should show values.
* Probably **no new correlation lines**: the aromatic HMBC/COSY correlations either hang off groups that
  hold only HMBC peaks or the HSQC multiplet (e.g. para-H → ortho C ends in M7), or have no merged 2D
  signal at all (meta–ortho and meta–para COSY, meta-H → C2).
* If JASON rejects the step with "more than one signal", the symmetric pairs need one shift per pair:
  comment out the C3, H3, C6 and H6 rows and run `run_step(4, tag="b")`.


In [45]:
run_step(4)

Step 4: 15 assignment(s)
  C8 carbon (CH3)      -> spectrum [1] at   15.7155  ok
  C2 carbon (quat)     -> spectrum [1] at  144.3054  ok
  H8 proton (CH3)      -> spectrum [6] at    1.3676  ok
  C7 carbon (CH2)      -> spectrum [1] at   28.9897  ok
  H7 proton (CH2)      -> spectrum [6] at    2.7775  ok
  C1 carbon (ortho)    -> spectrum [1] at  127.9292  ok
  C3 carbon (ortho)    -> spectrum [1] at  127.9292  ok
  H1 proton (ortho)    -> spectrum [6] at    7.3226  ok
  H3 proton (ortho)    -> spectrum [6] at    7.3226  ok
  C4 carbon (meta)     -> spectrum [1] at  128.3836  ok
  C6 carbon (meta)     -> spectrum [1] at  128.3836  ok
  H4 proton (meta)     -> spectrum [6] at    7.4012  ok
  H6 proton (meta)     -> spectrum [6] at    7.4012  ok
  C5 carbon (para)     -> spectrum [1] at  125.6658  ok
  H5 proton (para)     -> spectrum [6] at    7.2956  ok

Saved /Users/vsmw51/Documents/ethylbenzene_tests/ethylbenzene_step4.jjh5 - open it in JASON.


In [46]:
run_step(5)

  move HMBC meta-H -> C2: (7.4039, 144.4006, 0.0000) -> (7.4012, 144.3054, 0.0000)   ShapePar=[0. 0. 0.]
Step 5: 15 assignment(s)
  C8 carbon (CH3)      -> spectrum [1] at   15.7155  ok
  C2 carbon (quat)     -> spectrum [1] at  144.3054  ok
  H8 proton (CH3)      -> spectrum [6] at    1.3676  ok
  C7 carbon (CH2)      -> spectrum [1] at   28.9897  ok
  H7 proton (CH2)      -> spectrum [6] at    2.7775  ok
  C1 carbon (ortho)    -> spectrum [1] at  127.9292  ok
  C3 carbon (ortho)    -> spectrum [1] at  127.9292  ok
  H1 proton (ortho)    -> spectrum [6] at    7.3226  ok
  H3 proton (ortho)    -> spectrum [6] at    7.3226  ok
  C4 carbon (meta)     -> spectrum [1] at  128.3836  ok
  C6 carbon (meta)     -> spectrum [1] at  128.3836  ok
  H4 proton (meta)     -> spectrum [6] at    7.4012  ok
  H6 proton (meta)     -> spectrum [6] at    7.4012  ok
  C5 carbon (para)     -> spectrum [1] at  125.6658  ok
  H5 proton (para)     -> spectrum [6] at    7.2956  ok

Saved /Users/vsmw51/Documents

In [48]:
run_step(6)

  move HMBC meta-H -> C2: (7.4039, 144.4006, 0.0000) -> (7.4012, 144.3054, 0.0000)   ShapePar=[0. 0. 0.]
  move HMBC CH3 -> C2: (1.3680, 144.4002, 0.0000) -> (1.3676, 144.3054, 0.0000)   ShapePar=[0. 0. 0.]
  move HMBC CH3 -> C7: (1.3681, 28.9667, 0.0000) -> (1.3676, 28.9897, 0.0000)   ShapePar=[0. 0. 0.]
  move HMBC CH2 -> C2: (2.7748, 144.3940, 0.0000) -> (2.7775, 144.3054, 0.0000)   ShapePar=[0. 0. 0.]
  move HMBC CH2 -> ortho: (2.7752, 127.9209, 0.0000) -> (2.7775, 127.9292, 0.0000)   ShapePar=[0. 0. 0.]
  move HMBC para-H -> ortho: (7.2955, 127.9546, 0.0000) -> (7.2956, 127.9292, 0.0000)   ShapePar=[0. 0. 0.]
  move COSY CH2 - CH3: (2.7759, 1.3684, 0.0000) -> (2.7775, 1.3676, 0.0000)   ShapePar=[0. 0. 0.]
  move COSY meta - ortho: (7.4051, 7.3263, 0.0000) -> (7.4012, 7.3226, 0.0000)   ShapePar=[0. 0. 0.]
  move COSY meta - para: (7.4047, 7.2966, 0.0000) -> (7.4012, 7.2956, 0.0000)   ShapePar=[0. 0. 0.]
Step 6: 15 assignment(s)
  C8 carbon (CH3)      -> spectrum [1] at   15.7155  o